# Análise Exploratória — Brasileirão Série A

Análise dos dados do Campeonato Brasileiro Série A conectando diretamente ao PostgreSQL.

> **Pré-requisito:** inicie o Jupyter a partir da pasta `Banco de Dados/` para que o `.env` com as credenciais seja carregado automaticamente.
>
> ```bash
> cd "Banco de Dados"
> jupyter notebook
> ```

In [ ]:
import os
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv

%matplotlib inline
load_dotenv()

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

conn = psycopg2.connect(
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "5432")),
    dbname=os.getenv("DB_NAME", "brasileirao"),
    user=os.getenv("DB_USER", "brasileirao_user"),
    password=os.getenv("DB_PASSWORD", "brasileirao_pass"),
)


def run(sql):
    """Executa um SQL e retorna DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


print("Conectado ao PostgreSQL com sucesso!")

---
## Pergunta 1 — O Fator "Casa" por Estádio

**Pergunta:** Em quais estádios os times mandantes apresentam maior vantagem ofensiva?

**Requisitos SQL:** `SELECT`, `JOIN`, `GROUP BY`, `HAVING`, `AVG`, `COUNT`, `WHERE`.

In [ ]:
SQL_Q1 = """
SELECT
    e.nome_estadio,
    COUNT(p.id_partida)   AS total_jogos,
    AVG(p.publico)        AS media_publico,
    AVG(p.gols_mandante)  AS media_gols_mandante,
    AVG(p.gols_visitante) AS media_gols_visitante
FROM partida p
JOIN estadio e ON p.id_estadio = e.id_estadio
WHERE p.publico IS NOT NULL
GROUP BY e.nome_estadio
HAVING COUNT(p.id_partida) >= 10
ORDER BY media_gols_mandante DESC
LIMIT 15
"""

df_q1 = run(SQL_Q1)
df_q1[['media_gols_mandante', 'media_gols_visitante']] = (
    df_q1[['media_gols_mandante', 'media_gols_visitante']].astype(float)
)
display(df_q1)

# ── Gráfico de Barras Agrupadas ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(df_q1))
w = 0.38

ax.bar([i - w / 2 for i in x], df_q1['media_gols_mandante'], w,
       label='Mandante', color='#27ae60')
ax.bar([i + w / 2 for i in x], df_q1['media_gols_visitante'], w,
       label='Visitante', color='#c0392b', alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(df_q1['nome_estadio'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Média de Gols por Jogo')
ax.set_title(
    'Fator Casa por Estádio — Top 15 (mín. 10 jogos com público informado)',
    fontsize=12, fontweight='bold'
)
ax.legend()
plt.tight_layout()
plt.show()

### Discussão

Estádios com alto público nem sempre concentram a maior diferença de gols.
Arenas menores e mais intimistas podem gerar pressão local mais intensa, beneficiando o mandante mesmo com menos torcedores presentes.
Um estádio que aparece no topo com baixa `media_publico` é um candidato a essa anomalia — possivelmente um time com estilo de jogo agressivo em casa ou adversários que têm dificuldade em ambientes mais hostis.
Já grandes arenas modernas podem "diluir" a pressão ao acomodar torcidas mistas.

---
## Pergunta 2 — Eficiência Ofensiva vs. Valor do Elenco

**Pergunta:** Times com elencos mais caros são mais eficientes (precisam de menos chutes para marcar)?

**Requisitos SQL:** `CTE (WITH)`, `JOIN`, `SUM`, `CASE`, `NULLIF`, `ROUND`.

In [ ]:
SQL_Q2 = """
WITH EficienciaOfensiva AS (
    SELECT
        t.nome_time,
        AVG(est.valor_equipe_titular) AS valor_medio_elenco,
        SUM(est.chutes)               AS total_chutes,
        SUM(CASE
            WHEN est.tipo_mando = 'mandante' THEN p.gols_mandante
            ELSE p.gols_visitante
        END)                          AS total_gols
    FROM estatistica est
    JOIN partida p ON est.id_partida = p.id_partida
    JOIN time t    ON est.id_time    = t.id_time
    GROUP BY t.nome_time
)
SELECT
    nome_time,
    ROUND(valor_medio_elenco::NUMERIC, 2)                        AS valor_medio_elenco,
    total_gols,
    total_chutes,
    ROUND((total_chutes::NUMERIC / NULLIF(total_gols, 0)), 2)    AS chutes_por_gol
FROM EficienciaOfensiva
WHERE total_gols > 0
  AND valor_medio_elenco IS NOT NULL
  AND total_chutes IS NOT NULL
ORDER BY chutes_por_gol ASC
"""

df_q2 = run(SQL_Q2)
df_q2[['valor_medio_elenco', 'chutes_por_gol']] = (
    df_q2[['valor_medio_elenco', 'chutes_por_gol']].astype(float)
)
display(df_q2)

# ── Scatter Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 9))

sc = ax.scatter(
    df_q2['valor_medio_elenco'],
    df_q2['chutes_por_gol'],
    c=df_q2['chutes_por_gol'],
    cmap='RdYlGn_r',
    s=80,
    alpha=0.85,
    edgecolors='white',
    linewidths=0.5
)
plt.colorbar(sc, ax=ax, label='Chutes por Gol')

# Linha de mediana como referência
mediana = df_q2['chutes_por_gol'].median()
ax.axhline(mediana, color='gray', linestyle='--', alpha=0.6,
           label=f'Mediana: {mediana:.1f}')

# Rotula todos os pontos
for _, row in df_q2.iterrows():
    ax.annotate(
        row['nome_time'],
        (row['valor_medio_elenco'], row['chutes_por_gol']),
        textcoords='offset points', xytext=(6, 4), fontsize=6.5
    )

ax.set_xlabel('Valor Médio do Elenco Titular (€)')
ax.set_ylabel('Chutes por Gol  (↓ = mais eficiente)')
ax.set_title('Eficiência Ofensiva vs. Valor do Elenco', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'€{v:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

### Discussão

A tendência esperada é que times mais caros precisem de menos chutes para marcar (jogadores mais precisos e com melhor posicionamento).
Se no gráfico há um time de baixo valor mas com `chutes_por_gol` reduzido, isso é um sinal de excelência tática — possivelmente um time reativo que aproveita contra-ataques de forma muito eficaz.
O inverso também é revelador: elencos caros com alta `chutes_por_gol` podem indicar times que dominam a posse de bola mas têm baixa conversão, ou que enfrentaram goleiros em grande fase durante o período analisado.

---
## Pergunta 3 — O Perfil dos Árbitros

**Pergunta:** Quais árbitros registram as maiores médias de faltas, e como se comparam com a média geral?

**Requisitos SQL:** `Window Function (OVER)`, subconsulta, `COUNT(DISTINCT)`, `AVG` aninhado.

In [ ]:
SQL_Q3 = """
SELECT
    pes.nome                                                        AS nome_arbitro,
    COUNT(DISTINCT p.id_partida)                                    AS partidas_apitadas,
    ROUND(AVG(est_totais.total_faltas_partida)::NUMERIC, 2)         AS media_faltas_arbitro,
    ROUND((AVG(AVG(est_totais.total_faltas_partida)) OVER())::NUMERIC, 2)
        AS media_geral_campeonato
FROM partida p
JOIN arbitro arb ON p.id_arbitro  = arb.id_arbitro
JOIN pessoa  pes ON arb.id_arbitro = pes.id_pessoa
JOIN (
    SELECT id_partida, SUM(faltas) AS total_faltas_partida
    FROM estatistica
    WHERE faltas IS NOT NULL
    GROUP BY id_partida
) est_totais ON p.id_partida = est_totais.id_partida
GROUP BY pes.nome
HAVING COUNT(DISTINCT p.id_partida) >= 5
ORDER BY media_faltas_arbitro DESC
LIMIT 20
"""

df_q3 = run(SQL_Q3)
df_q3[['media_faltas_arbitro', 'media_geral_campeonato']] = (
    df_q3[['media_faltas_arbitro', 'media_geral_campeonato']].astype(float)
)
display(df_q3)

# ── Gráfico de Barras Horizontal ─────────────────────────────────────────────
media_geral = df_q3['media_geral_campeonato'].iloc[0]

# Barras vermelhas para árbitros acima da média, azuis para abaixo
cores = ['#e74c3c' if v > media_geral else '#3498db'
         for v in df_q3['media_faltas_arbitro']]

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(df_q3['nome_arbitro'], df_q3['media_faltas_arbitro'],
        color=cores, alpha=0.85)
ax.axvline(media_geral, color='black', linestyle='--', linewidth=1.5,
           label=f'Média geral: {media_geral:.2f}')

ax.set_xlabel('Média de Faltas por Jogo')
ax.set_title(
    'Perfil dos Árbitros — Média de Faltas (mín. 5 partidas, top 20)',
    fontsize=12, fontweight='bold'
)
ax.invert_yaxis()  # árbitro com mais faltas no topo
ax.legend()
plt.tight_layout()
plt.show()

### Discussão

Árbitros muito acima da média marcam praticamente qualquer contato, "picotando" o jogo e reduzindo o tempo efetivo de bola rolando.
Esse padrão pode ser sistêmico (árbitro sempre assim) ou episódico (um determinado par de times mais agressivos).
Registros zerados ou muito baixos de faltas são um sinal de alerta para **qualidade dos dados**: na base do Transfermarkt, a súmula de algumas partidas antigas pode estar incompleta, especialmente para anos como 2005–2010.
Uma análise complementar seria cruzar esse padrão com o número de cartões, para verificar se árbitros de alta falta também aplicam mais punições.

---
## Pergunta 4 — Juventude × Experiência na Tabela

**Pergunta:** Existe correlação entre a idade média do time titular e a colocação final no campeonato?

**Requisitos SQL:** `MIN`, `MAX`, `AVG`, `GROUP BY`, `WHERE`, `ORDER BY`.

In [ ]:
SQL_Q4 = """
SELECT
    est.colocacao,
    ROUND(AVG(est.idade_media_titular)::NUMERIC, 2) AS idade_media_geral,
    MIN(est.idade_media_titular)                    AS time_mais_jovem,
    MAX(est.idade_media_titular)                    AS time_mais_velho
FROM estatistica est
WHERE est.colocacao IS NOT NULL
GROUP BY est.colocacao
ORDER BY est.colocacao ASC
"""

df_q4 = run(SQL_Q4)
df_q4 = df_q4.astype(float)
display(df_q4)

# ── Line Chart com banda min–máx ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df_q4['colocacao'], df_q4['idade_media_geral'],
        marker='o', linewidth=2, color='#2980b9', label='Idade média')
ax.fill_between(
    df_q4['colocacao'], df_q4['time_mais_jovem'], df_q4['time_mais_velho'],
    alpha=0.15, color='#2980b9', label='Intervalo (min – máx)'
)

# Linhas de referência para G4 e zona de rebaixamento (4 times rebaixados)
ax.axvline(4.5,  color='green', linestyle=':', linewidth=1.5, label='Corte G4')
ax.axvline(16.5, color='red',   linestyle=':', linewidth=1.5, label='Zona de rebaixamento')

ax.set_xlabel('Colocação Final no Campeonato')
ax.set_ylabel('Idade Média dos Titulares (anos)')
ax.set_title(
    'Juventude × Experiência — Idade Média por Colocação Final',
    fontsize=12, fontweight='bold'
)
ax.set_xticks(df_q4['colocacao'].astype(int).tolist())
ax.legend()
plt.tight_layout()
plt.show()

### Discussão

A tendência esperada em campeonatos de pontos corridos é que o **G4** (posições 1–4) apresente times ligeiramente mais experientes, pois a consistência ao longo de 38 rodadas favorece elencos maduros.
Se o gráfico mostra o campeão com idade média alta, isso reforça a hipótese da experiência.
O padrão oposto — times na zona de rebaixamento com titulares jovens — pode indicar clubes que perderam jogadores sênior por questões financeiras e recorreram às categorias de base por necessidade, não por escolha estratégica.
A banda min–máx larga em determinadas colocações indica alta variância: diferentes times chegaram à mesma posição com perfis etários muito distintos.

---
## Conclusão

As quatro análises revelam padrões distintos e complementares:

| Análise | Principal achado esperado |
|---|---|
| Fator casa | Estádios menores podem ter maior vantagem relativa |
| Eficiência ofensiva | Elenco caro ≠ garantia de alta conversão |
| Perfil de árbitros | Estilos de apitação sistematicamente diferentes |
| Juventude vs. experiência | G4 levemente mais experiente; rebaixados mais jovens |

### Conexão com a Lei de Acesso à Informação (LAI) e Transparência Esportiva

A abertura dos dados estatísticos do Brasileirão permite que **imprensa independente, torcedores e pesquisadores** auditem o campeonato sem depender de versões oficiais das entidades organizadoras.
Análises como a do perfil dos árbitros (Pergunta 3) são especialmente sensíveis: com dados abertos, é possível verificar se um árbitro específico prejudica sistematicamente determinado time — algo que seria impossível de detectar sem acesso às súmulas em formato estruturado.
Esse tipo de uso da informação pública promove a **transparência esportiva** e exemplifica como os princípios da LAI se estendem além do setor governamental, alcançando entidades que administram competições de interesse público.

In [ ]:
conn.close()
print("Conexão encerrada.")